# Census OCR — Colab + Qwen2.5-VL

**Start at Section 3** — upload mode needs no zip and no Drive.


## 1. Setup

In [1]:
!pip install -q "transformers>=4.49" accelerate bitsandbytes qwen-vl-utils openpyxl pandas fuzzywuzzy python-levenshtein matplotlib einops


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 75.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.7 MB/s eta 0:00:00


In [2]:
import torch
VRAM_GB = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
print(f'VRAM: {VRAM_GB:.1f} GB')
MODEL_ID="Qwen/Qwen2.5-VL-32B-Instruct"
FALLBACK="Qwen/Qwen2.5-VL-7B-Instruct"
LOAD_IN_4BIT=True
MAX_NEW_TOKENS=4096
MAX_PIXELS=1280*28*28
SELECTED_MODEL=MODEL_ID if VRAM_GB>=20 else FALLBACK
print('Model:', SELECTED_MODEL)


VRAM: 85.1 GB
Model: Qwen/Qwen2.5-VL-32B-Instruct


In [3]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True) if LOAD_IN_4BIT else None
print(f"Loading {SELECTED_MODEL}...")
processor = AutoProcessor.from_pretrained(SELECTED_MODEL, trust_remote_code=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    SELECTED_MODEL, quantization_config=bnb, device_map="auto",
    torch_dtype=torch.bfloat16, trust_remote_code=True)
model.eval()
print("Model ready.")


Loading Qwen/Qwen2.5-VL-32B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

...

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1161 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model ready.


## 3. Get your files into Colab

### Option 1: UPLOAD (do this first)
1. Run next cell (`DATA_MODE = "upload"` is already set)
2. Click **Choose Files** → select `sheet_01.jpg` from your Mac:
   `Downloads/poc_textual/data/raw_images/1950_11-1/sheet_01.jpg`
3. Click **Choose Files** again → select:
   `Downloads/poc_textual/data/ground_truth/Bastrop County 1950 Clean.xlsx`

### Option 2: Google Drive (all 11 sheets)
1. Open https://drive.google.com in browser
2. Drag the `poc_textual` folder from Downloads onto Drive
3. In next cell change to `DATA_MODE = "drive"`, run it, authorize
4. If folder is not at My Drive/poc_textual, edit PROJECT_DIR in that cell

### Option 3: Zip on Drive
Upload `poc_textual_colab.zip` to Google Drive (Drive accepts zips!).
Set `DATA_MODE = "drive_zip"` in next cell.


In [ ]:
from pathlib import Path
import shutil

DATA_MODE = "upload"  # upload | drive | drive_zip

PROJECT_DIR = Path("/content/poc_textual")
GT_PATH = PROJECT_DIR / "data/ground_truth/Bastrop County 1950 Clean.xlsx"
IMAGE_DIR = PROJECT_DIR / "data/raw_images/1950_11-1"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
(GT_PATH.parent).mkdir(parents=True, exist_ok=True)

if DATA_MODE == "upload":
    from google.colab import files
    print(">>> Pick ONE census image (e.g. sheet_01.jpg)")
    img_up = files.upload()
    img_name = list(img_up.keys())[0]
    (IMAGE_DIR / img_name).write_bytes(img_up[img_name])
    IMAGE_FILENAME = img_name
    print("  OK:", IMAGE_DIR / img_name)
    print(">>> Pick ground truth Excel (Bastrop County 1950 Clean.xlsx)")
    gt_up = files.upload()
    gt_name = list(gt_up.keys())[0]
    GT_PATH = PROJECT_DIR / "data/ground_truth" / gt_name
    GT_PATH.parent.mkdir(parents=True, exist_ok=True)
    GT_PATH.write_bytes(gt_up[gt_name])
    RUN_BATCH = False
    print("  OK:", GT_PATH)
    print("Done. Keep RUN_BATCH=False for first test.")

elif DATA_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/poc_textual")
    GT_PATH = PROJECT_DIR / "data/ground_truth/Bastrop County 1950 Clean.xlsx"
    IMAGE_DIR = PROJECT_DIR / "data/raw_images/1950_11-1"
    if not PROJECT_DIR.exists():
        print("ERROR: folder not found:", PROJECT_DIR)
        print("Upload poc_textual to Drive first. Searching...")
        for p in Path("/content/drive/MyDrive").rglob("Bastrop County 1950 Clean.xlsx"):
            print("  Found ground truth at:", p.parent.parent.parent)
    else:
        nj = len(list(IMAGE_DIR.glob("*.jpg"))) if IMAGE_DIR.exists() else 0
        print("OK | xlsx:", GT_PATH.exists(), "| jpg files:", nj)
    RUN_BATCH = True

elif DATA_MODE == "drive_zip":
    from google.colab import drive
    drive.mount("/content/drive")
    z = Path("/content/drive/MyDrive/poc_textual_colab.zip")
    if not z.exists():
        raise FileNotFoundError("Upload poc_textual_colab.zip to Google Drive (My Drive root)")
    shutil.unpack_archive(str(z), "/content", "zip")
    PROJECT_DIR = Path("/content/poc_textual")
    GT_PATH = PROJECT_DIR / "data/ground_truth/Bastrop County 1950 Clean.xlsx"
    IMAGE_DIR = PROJECT_DIR / "data/raw_images/1950_11-1"
    RUN_BATCH = True
    print("Unzipped. Images:", len(list(IMAGE_DIR.glob("*.jpg"))))

print("GT_PATH =", GT_PATH)
print("IMAGE_DIR =", IMAGE_DIR)


>>> Pick ONE census image (e.g. sheet_01.jpg)


## 3. Configure the run

Set the census year and the exact ground-truth sheet name to compare against.

**Verified-clean 1950 sheets (no known typos):** `Bastrop 11-3`, `Bastrop 11-2A`,
`Smithville 11-10`, `Elgin 11-20`, `Elgin 11-21`.
Avoid `Bastrop all Manipulated`, `Bastrop 11-2B`, `Bastrop 11-1`,
`Smithville 11-7`, `Smithville 11-8`, `Smithville 11-9` for now — these contain
stray/typo values in their Race column that are ground-truth data-entry
mistakes, not extraction errors.

In [ ]:
CENSUS_YEAR=1950
GROUND_TRUTH_SHEET="Bastrop 11-1"
ENUMERATION_DISTRICT="11-1"
PHYSICAL_PAGE=1
if "IMAGE_FILENAME" not in dir(): IMAGE_FILENAME="sheet_01.jpg"
if "RUN_BATCH" not in dir(): RUN_BATCH=False
BATCH_PAGES=list(range(1,12))
print("page",PHYSICAL_PAGE,"image",IMAGE_FILENAME,"batch",RUN_BATCH)


## 4. Schema & normalization utilities

Verified directly against the ground truth files -- see `CLAUDE.md` for the full investigation. **Do not treat these as textbook census categories** — several are a research-team-specific controlled vocabulary that a vision model reading the raw image cannot produce directly (e.g. "Negro (Black)", "Mexican (Latino)", and especially the 1950 "W0"/"WO" sub-code, which is not visible anywhere on the physical form).

In [ ]:
BIRTHPLACE_COLUMN = {
    1850: "Birth Place", 1860: "Birth Place", 1950: "Birth Place",
    1870: "Birthplace", 1880: "Birthplace", 1900: "Birthplace",
    1910: "Birthplace", 1920: "Birthplace", 1930: "Birthplace", 1940: "Birthplace",
}

GENDER_COLUMN = {1920: "Sex"}  # every other decade uses "Gender"

HAS_LINE_NUMBER = {
    1850: True, 1860: False,  # 1860 confirmed to have NO Line Number column anywhere
    1870: True, 1880: True, 1900: True, 1910: True,
    1920: True, 1930: True, 1940: True, 1950: True,
}

# Race values ACTUALLY OBSERVED in the ground truth per decade (verified by
# scanning every sheet in every workbook -- not textbook categories)
VALID_RACE = {
    1850: ["White", "Black", "Mulatto"],
    1860: ["White", "Black", "Mulatto", "Indian (Native American)"],
    1870: ["White", "Black", "Mulatto"],
    1880: ["White", "Black", "Mulatto", "Filipino"],
    1900: ["White", "Black", "Mulatto", "Chinese", "Mexican (Latino)"],
    1910: ["White", "Black", "Mulatto", "Octoroon", "Mexican (Latino)", "Other"],
    1920: ["White", "Black", "Mulatto", "Mexican (Latino)"],
    1930: ["White", "Black", "Negro (Black)", "Mulatto", "Mexican (Latino)"],
    1940: ["White", "Negro (Black)"],
    1950: ["White", "Negro (Black)", "Chinese", "W0", "WO"],
}

# Raw form code (what a vision model reads off the image) -> ground truth string
RACE_NORMALIZE_MAP = {
    1950: {"W": "White", "Neg": "Negro (Black)", "Ch": "Chinese"},
    1940: {"W": "White", "Ne": "Negro (Black)", "Neg": "Negro (Black)"},
    1930: {"W": "White", "B": "Black", "Mu": "Mulatto", "Mex": "Mexican (Latino)", "Neg": "Negro (Black)"},
    1920: {"W": "White", "B": "Black", "Mu": "Mulatto", "Mex": "Mexican (Latino)"},
    1910: {"W": "White", "B": "Black", "Mu": "Mulatto", "Ot": "Octoroon", "Mex": "Mexican (Latino)"},
    1900: {"W": "White", "B": "Black", "Mu": "Mulatto", "Ch": "Chinese", "Mex": "Mexican (Latino)"},
    1880: {"W": "White", "B": "Black", "Mu": "Mulatto", "Fil": "Filipino"},
    1870: {"W": "White", "B": "Black", "Mu": "Mulatto"},
    1860: {"W": "White", "B": "Black", "Mu": "Mulatto", "In": "Indian (Native American)"},
    1850: {"W": "White", "B": "Black", "Mu": "Mulatto"},
    # "W0"/"WO" (1950) intentionally excluded -- not derivable from the image, see CLAUDE.md
}

VALID_GENDER = ["Male", "Female"]

VALID_MARITAL_STATUS = {
    1880: ["Married", "Single", "Widowed", "Widower", "Divorced", "Na"],
    1900: ["Married", "Single", "Widowed", "Divorced"],
    1910: ["Married", "Single", "Widowed", "Divorced"],
    1920: ["Married", "Single", "Widowed", "Divorced"],
    1930: ["Married", "Single", "Widowed", "Divorced"],
    1940: ["Married", "Single", "Widowed", "Divorced"],
    1950: ["Married", "Never Married (Single)", "Widowed", "Divorced", "Separated"],
}


def normalize_race(raw, year):
    if not raw:
        return raw
    raw = raw.strip()
    decade_map = RACE_NORMALIZE_MAP.get(year, {})
    if raw in decade_map:
        return decade_map[raw]
    for v in VALID_RACE.get(year, []):
        if raw.lower() == v.lower():
            return v
    return raw


def normalize_gender(raw):
    if not raw:
        return raw
    raw = raw.strip().upper()
    if raw in ["M", "MALE"]:
        return "Male"
    if raw in ["F", "FEMALE"]:
        return "Female"
    return raw


def propagate_dittos(records, surname_field="Surname", birthplace_field="Birthplace"):
    prev_surname, prev_birthplace = None, None
    ditto_markers = [None, "", "\u2014\u2014", '"', "''", "ditto", "do", "Do", "DO"]
    for rec in records:
        surname = rec.get(surname_field)
        if surname in ditto_markers:
            if prev_surname:
                rec[surname_field] = prev_surname
        else:
            prev_surname = surname
        birthplace = rec.get(birthplace_field)
        if birthplace in ditto_markers:
            if prev_birthplace:
                rec[birthplace_field] = prev_birthplace
        else:
            prev_birthplace = birthplace
    return records

print("Utilities loaded.")

# Maps the raw abbreviation a vision model reads off the form (Mar, Wd, D, S...)
# to the ground-truth string for that decade. CONFIRMED NECESSARY: fuzzy string
# matching fails badly on these (e.g. "mar" vs "married" scores only 60/100,
# "d" vs "divorced" scores 22/100, both well below any reasonable threshold) --
# without this normalization, correct extractions get marked as wrong.
MARITAL_STATUS_NORMALIZE_MAP = {
    1950: {"Mar": "Married", "Wd": "Widowed", "D": "Divorced", "Sep": "Separated",
           "S": "Never Married (Single)"},
    "default": {"Mar": "Married", "M": "Married", "Wd": "Widowed", "W": "Widowed",
                "D": "Divorced", "S": "Single", "Sep": "Separated"},
}


def normalize_marital_status(raw, year):
    if not raw:
        return raw
    raw = raw.strip()
    decade_map = MARITAL_STATUS_NORMALIZE_MAP.get(year, MARITAL_STATUS_NORMALIZE_MAP["default"])
    if raw in decade_map:
        return decade_map[raw]
    for v in VALID_MARITAL_STATUS.get(year, []):
        if raw.lower() == v.lower():
            return v
    return raw


## 5. Extraction prompt for this decade

Uses the verified schema and instructs the model to transcribe raw marks rather than expand or normalize them (normalization happens separately, below).

In [ ]:
SYSTEM_PROMPT = """You are a specialized historical census transcription assistant. You read
handwritten U.S. Census record images and extract structured data precisely.

RULES:
1. Transcribe EXACTLY what is written. Do not correct spelling of names.
2. If a field is blank or empty, use null.
3. If handwriting is truly illegible after careful inspection, use "[illegible]". Do NOT guess.
4. Ditto marks (\u2014\u2014, ", ditto) mean "same as the row above" -- write out the actual
   value, do not write the ditto mark itself.
5. "No one at home", "Vacant", or similar -- include as a record with line_number filled
   and all person fields null.
6. Return ONLY valid JSON. No explanation, no markdown fences, no preamble.
7. Race and marital status: transcribe the LITERAL mark on the page as written (e.g. "W"
   stays "W", "Neg" stays "Neg"). Do NOT expand abbreviations or map to any external
   category system -- normalization happens downstream, not here.
8. Gender/Sex: write "Male" or "Female" (safe to expand -- consistent across decades).
"""

PROMPTS_1950 = """This is a page from the 1950 U.S. Census of Population and Housing (Form P1).
State: Texas, County: Bastrop.

The form has two sections:
  - MAIN RECORDS (lines 1-30): extract all of these
  - SAMPLE LINES (bottom section, separate grid): label these with line_number as "S1", "S2", etc.

For each numbered line in MAIN RECORDS, extract using EXACTLY these key names:
  "Line Number"                integer
  "Street Name"                string or null
  "House Number"               integer or null
  "Dwelling Number"            integer or null
  "Surname"                    string or null (written once per household; propagate for ditto marks)
  "Given Name"                 string or null
  "Relation to Head of House"  string (Head, Wife, Son, Daughter, Lodger, etc.)
  "Race"                       string -- transcribe EXACTLY what letter/word is on the form (e.g. "W", "Neg")
  "Gender"                     string (Male or Female)
  "Age"                        integer or null
  "Marital Status"             string as abbreviated on form (Mar, Wd, D, Sep, S) or null
  "Birth Place"                string (state or country) or null
  "Occupation"                 string or null
  "Industry"                   string or null
  "Worker Class"               string (P, G, O, NP as marked) or null

Return a JSON array of objects, one per line.

NOTE: This decade's ground truth also uses a sub-code ("W0"/"WO") applied to some
White-coded individuals based on Spanish-language surnames. That code is NOT visible
anywhere on the physical form -- do not attempt to guess it. Just transcribe what's written.
"""

DECADE_PROMPTS = {1950: PROMPTS_1950}  # add other decades from PROMPTS.md as needed

if CENSUS_YEAR not in DECADE_PROMPTS:
    raise ValueError(
        f"No prompt defined in this notebook for {CENSUS_YEAR}. "
        f"Copy the matching block from PROMPTS.md in the project repo."
    )
print(f"Prompt ready for {CENSUS_YEAR}.")

## 6. Run extraction

In [ ]:
import json
import re
import torch
from qwen_vl_utils import process_vision_info

def decade_prompt(y,ed):
    return PROMPTS_1950.replace('{ED}', ed)

def parse_json_records(raw):
    raw = re.sub(r"```json\s*", "", raw.strip())
    raw = re.sub(r"```\s*$", "", raw, flags=re.MULTILINE).strip()
    s, e = raw.find("["), raw.rfind("]")
    if s >= 0 and e > s:
        raw = raw[s:e + 1]
    return json.loads(raw)

def extract_from_image(image_path, year, ed=ENUMERATION_DISTRICT):
    full = SYSTEM_PROMPT + "\n\n" + decade_prompt(year, ed)
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": str(image_path), "max_pixels": MAX_PIXELS},
        {"type": "text", "text": full},
    ]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(msgs)
    inp = processor(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trim = [o[len(i):] for i, o in zip(inp.input_ids, gen)]
    raw = processor.batch_decode(trim, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    print("--- model raw output (first 500 chars) ---")
    print(raw[:500])
    recs = parse_json_records(raw)
    bp = BIRTHPLACE_COLUMN.get(year, "Birthplace")
    recs = propagate_dittos(recs, birthplace_field=bp)
    gf = GENDER_COLUMN.get(year, "Gender")
    for r in recs:
        if "Race" in r:
            r["Race"] = normalize_race(r["Race"], year)
        if "Marital Status" in r:
            r["Marital Status"] = normalize_marital_status(r["Marital Status"], year)
        if gf in r:
            r[gf] = normalize_gender(r[gf])
        elif "Gender" in r:
            r["Gender"] = normalize_gender(r["Gender"])
    return recs

image_path = IMAGE_DIR / IMAGE_FILENAME
print("Extracting", image_path)
extracted_records = extract_from_image(image_path, CENSUS_YEAR)
print(len(extracted_records), "records")
extracted_records[:3]


## 7. Compare against ground truth

Field-matching strategy is chosen by keyword, not a fixed column list, since column names genuinely differ by decade (`"Birthplace"` vs `"Birth Place"`, `"Gender"` vs `"Sex"`, etc. -- verified, see CLAUDE.md).

**Important:** a ground-truth sheet like `Bastrop 11-2A` is not one physical census page -- it's dozens of physical pages concatenated together, with Line Number resetting to 1 at the start of each one (confirmed: 24 separate pages inside `Bastrop 11-2A` alone). You must tell `compare()` which physical page to check against via `PHYSICAL_PAGE` below, or you will silently get matched against the wrong data.

In [ ]:
COMPARE_FIELDS_1950 = ['Street Name', 'House Number', 'Dwelling Number', 'Surname', 'Given Name', 'Relation to Head of House', 'Race', 'Gender', 'Age', 'Marital Status', 'Birth Place', 'Occupation', 'Industry', 'Worker Class']
PRIORITY_FIELDS_1950 = ['Race', 'Gender', 'Surname', 'Given Name', 'Age', 'Relation to Head of House', 'Birth Place']
from fuzzywuzzy import fuzz
import pandas as pd
STRATEGY_KEYWORDS = [(['race'],'exact'),(['gender','sex'],'exact'),(['marital status'],'exact'),
 (['age'],'numeric'),(['surname','given name'],'fuzzy_name'),(['relation'],'fuzzy'),
 (['birthplace','birth place'],'fuzzy'),(['occupation','industry'],'fuzzy')]
def classify_strategy(field):
 fl=field.lower()
 for kws,st in STRATEGY_KEYWORDS:
  if any(k in fl for k in kws): return st
 return 'fuzzy'
def _norm(v):
 if v is None or (isinstance(v,float) and pd.isna(v)): return ''
 return str(v).strip().lower()
def field_match(ext,gt,strategy):
 e,g=_norm(ext),_norm(gt)
 if e==g=='': return True
 if e=='' or g=='': return False
 if strategy=='exact': return e==g
 if strategy in ('fuzzy','fuzzy_name'): return fuzz.ratio(e,g)>=(90 if strategy=='fuzzy_name' else 85)
 if strategy=='numeric':
  try: return abs(int(float(e))-int(float(g)))<=1
  except: return e==g
 return e==g
def split_into_physical_pages(gt_df,line_col='Line Number'):
 lines=pd.to_numeric(gt_df[line_col],errors='coerce'); starts=[0]
 for i in range(1,len(lines)):
  if pd.notna(lines.iloc[i]) and lines.iloc[i]==1 and (pd.isna(lines.iloc[i-1]) or lines.iloc[i-1]!=1): starts.append(i)
 starts.append(len(gt_df)); return [gt_df.iloc[s:e].reset_index(drop=True) for s,e in zip(starts,starts[1:])]
def compare(records,gt_xlsx_path,gt_sheet,year,physical_page):
 gt_df=pd.read_excel(gt_xlsx_path,sheet_name=gt_sheet); pages=split_into_physical_pages(gt_df)
 page_df=pages[physical_page-1]
 gt_by_line={int(r['Line Number']):r for _,r in page_df.iterrows() if pd.notna(r.get('Line Number'))}
 ext_by_line={int(r['Line Number']):r for r in records if r.get('Line Number') is not None}
 cols=[c for c in COMPARE_FIELDS_1950 if c in page_df.columns]
 print(f'Comparing {len(cols)} fields (not all {len(page_df.columns)-1} GT cols)')
 results,field_scores=[],{c:[] for c in cols}; priority_scores={c:[] for c in cols if c in PRIORITY_FIELDS_1950}
 for ln in sorted(gt_by_line.keys()):
  gt_row,ext_row=gt_by_line[ln],ext_by_line.get(ln,{}); row={'line_number':ln,'all_match':True,'fields':{}}
  for f in cols:
   m=field_match(ext_row.get(f),gt_row.get(f),classify_strategy(f)); row['fields'][f]={'extracted':ext_row.get(f),'ground_truth':gt_row.get(f),'match':m}; field_scores[f].append(m)
   if f in priority_scores: priority_scores[f].append(m)
   if not m: row['all_match']=False
  results.append(row)
 fa={f:sum(s)/len(s) for f,s in field_scores.items() if s}; pa={f:sum(s)/len(s) for f,s in priority_scores.items() if s}
 return {'rows_compared':len(results),'row_accuracy':sum(r['all_match'] for r in results)/len(results) if results else 0,'field_accuracy':fa,'overall_field_accuracy':sum(fa.values())/len(fa) if fa else 0,'priority_field_accuracy':sum(pa.values())/len(pa) if pa else 0,'physical_page':physical_page,'sheet':gt_sheet,'census_year':year,'total_physical_pages_in_sheet':len(pages)}, results
print('compare() defined')


In [ ]:
gt_path = GT_PATH
metrics, results = compare(extracted_records, gt_path, GROUND_TRUTH_SHEET, CENSUS_YEAR, PHYSICAL_PAGE)
print(f"Rows compared: {metrics['rows_compared']}")
print(f"Row accuracy:  {metrics['row_accuracy']:.1%}")
print(f"Field accuracy: {metrics['overall_field_accuracy']:.1%}")


## 8. Visualize field-level accuracy

In [ ]:
import matplotlib.pyplot as plt

scored_fields = {k: v for k, v in metrics["field_accuracy"].items() if v is not None}
sorted_fields = dict(sorted(scored_fields.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(9, max(4, len(sorted_fields) * 0.4)))
bars = ax.barh(list(sorted_fields.keys()), [v * 100 for v in sorted_fields.values()])
ax.set_xlabel("Accuracy (%)")
ax.set_xlim(0, 100)
ax.set_title(f"Field-Level Extraction Accuracy \u2014 {metrics['census_year']} Census, sheet '{metrics['sheet']}'")
ax.axvline(x=metrics["overall_field_accuracy"] * 100, color="red", linestyle="--",
           label=f"Overall: {metrics['overall_field_accuracy']:.1%}")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Inspect mismatches

Most useful step for deciding what to fix next: real extraction errors vs. known ground-truth limitations (like the 1950 W0/WO sub-code, or typos in certain sheets — see CLAUDE.md).

In [ ]:
PRIORITY = PRIORITY_FIELDS_1950
mismatches=[]
for r in results:
 for field,d in r['fields'].items():
  if field not in PRIORITY or d['match']: continue
  mismatches.append({'line':r['line_number'],'field':field,'extracted':d['extracted'],'ground_truth':d['ground_truth']})
mismatch_df=pd.DataFrame(mismatches); print(f'Priority mismatches: {len(mismatch_df)}'); display(mismatch_df.head(30))


## 10. Save results

In [ ]:
import json

with open("poc_results.json", "w") as f:
    json.dump({"metrics": metrics, "results": results}, f, indent=2)

from google.colab import files
files.download("poc_results.json")
print("Saved and downloading poc_results.json")

## Notes on interpreting your accuracy numbers

- **Race field accuracy will not reach 100% on 1950 data** even with a perfect
  extractor. The ground truth's "W0"/"WO" sub-code is applied by the research
  team based on surname, not written anywhere on the physical census form —
  see CLAUDE.md section "VERIFIED Data Quality Findings" for the full
  investigation. Don't chase this to zero; flag it as a known, documented gap
  when you present results to Jaden.
- If you compare against a sheet other than `Bastrop 11-2A`, `Bastrop 11-3`,
  `Smithville 11-10`, `Elgin 11-20`, or `Elgin 11-21`, you may see stray
  ground-truth values (`'Whiite'`, `'White0'`, `'1'`, `'72'`, `'S'`, etc.) that
  are typos in the "clean" file itself, not extraction errors.
- To compare against Ancestry's own baseline OCR accuracy (to show the delta
  your pipeline achieves), use the paired Raw/Clean sheets in
  `Raw-Clean Comparison Document.xlsx` the same way — swap `gt_path` for that
  file and pick a `*_Raw` vs `*_Clean` sheet pair.

## Batch ED 11-1


In [ ]:
if RUN_BATCH:
 batch=[]; out=PROJECT_DIR/'results/colab_qwen'; out.mkdir(parents=True,exist_ok=True)
 for page in BATCH_PAGES:
  img=IMAGE_DIR/f'sheet_{page:02d}.jpg'
  if not img.exists(): continue
  recs=extract_from_image(img,CENSUS_YEAR); m,res=compare(recs,GT_PATH,GROUND_TRUTH_SHEET,CENSUS_YEAR,page)
  batch.append({'page':page,'row':m['row_accuracy'],'field':m['overall_field_accuracy']}); print(page,m['row_accuracy'],m['overall_field_accuracy'])
 if batch: json.dump({'model':SELECTED_MODEL,'pages':batch},open(out/'batch_summary.json','w'),indent=2)
